In [33]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import transformers
import warnings
transformers.logging.set_verbosity_error()
warnings.filterwarnings("ignore")

#data preprocessing
df = pd.read_csv('data/coffee_analysis.csv')
df = df.rename(columns={"loc_country": "region"})
geo_errors = ["Hawai'i", "Taiwan", "United States"]
df_cleaned = df.dropna(subset=['roast', 'desc_1']).copy()
df_cleaned = df_cleaned[df_cleaned['roast'].str.strip().str.len() > 0]
df_cleaned = df_cleaned[~df_cleaned['roast'].isin(geo_errors)]
df_cleaned = df_cleaned.reset_index(drop=True)

def simple_preprocess(text):
    if pd.isna(text): return ""
    return " ".join(re.findall(r'\w+', str(text).lower()))

df_cleaned['clean_desc'] = df_cleaned['desc_1'].apply(simple_preprocess)

#TF-IDF
tfidf_vec = TfidfVectorizer(stop_words='english', min_df=1)
tfidf_matrix = tfidf_vec.fit_transform(df_cleaned['clean_desc'])

all_roasters = df_cleaned['roaster'].dropna().unique()
all_origins = set(df_cleaned['origin_1'].dropna().unique()) | set(df_cleaned['origin_2'].dropna().unique())


In [34]:
def extract_price_from_query(query):

    patterns = [
        #  Keywords + Number + Optional Unit ("under 15 per 100g")
        r"(?:under|below|less than|budget of)\s?\$?(\d+(?:\.\d+)?)\s?(?:dollars|usd|per 100g|per 100 grams)?",
        # Number + Mandatory Unit ("15 dollars")
        r"\$?(\d+(?:\.\d+)?)\s?(?:dollars|usd|bucks|per 100g)"
    ]

    for pattern in patterns:
        match = re.search(pattern, query.lower())
        if match:
            price_value = float(match.group(1))
            matched_text = match.group(0)
            return price_value, matched_text

    return None, None

def extract_roast_from_query(query):

    roast_patterns = {
        'Light': r"\blight\s?(?:roast)?\b",
        'Medium-Light': r"\bmedium-light\s?(?:roast)?\b",
        'Medium': r"\bmedium\s?(?:roast)?\b",
        'Medium-Dark': r"\bmedium-dark\s?(?:roast)?\b",
        'Dark': r"\bdark\s?(?:roast)?\b"
    }

    for level, pattern in roast_patterns.items():
        match = re.search(pattern, query.lower())
        if match:
            return level, match.group(0)
    return None, None

**Baseline model**

In [35]:
def baseline_search(full_query, top_n=5):
    '''
    Hybrid Recommendation Engine (Baseline - TF-IDF version).

    Logic:
    1. Extract Hard Constraints (Price, Roaster, Origin, Roast Level) from the natural language query.
    2. Apply Hard Filters using Boolean Masking on the dataframe.
    3. Perform Soft Matching on flavor descriptions (desc_1) using TF-IDF and Cosine Similarity.
    4. Return Top-N recommendations sorted by Similarity and Rating.
    '''
    query_lower = full_query.lower()


    max_price, p_text = extract_price_from_query(query_lower)
    roast_level, r_text = extract_roast_from_query(query_lower)

    clean_query = query_lower
    if p_text: clean_query = clean_query.replace(p_text, "")
    if r_text: clean_query = clean_query.replace(r_text, "")
    clean_query = clean_query.strip()


    filters = {'roaster': None, 'origin': None}
    for r in all_roasters:
        if r.lower() in clean_query:
            filters['roaster'] = r
            break
    for o in all_origins:
        if len(o) > 3 and o.lower() in clean_query:
            filters['origin'] = o
            break

    print(f" [Baseline Log] Entities: {filters} | Price: {max_price} | Roast: {roast_level}")

    # Hard Filtering
    mask = pd.Series([True] * len(df_cleaned))
    if filters['roaster']:
        mask &= df_cleaned['roaster'].str.contains(filters['roaster'], case=False, regex=False)
    if filters['origin']:
        mask &= (df_cleaned['origin_1'].str.contains(filters['origin'], case=False, regex=False) |
                 df_cleaned['origin_2'].str.contains(filters['origin'], case=False, regex=False))
    if max_price:
        mask &= (df_cleaned['100g_USD'] <= max_price)
    if roast_level:
        mask &= (df_cleaned['roast'] == roast_level)

    df_subset = df_cleaned[mask].copy()
    if df_subset.empty:
        return " No matching results found for these specific constraints."

    #Soft Matching
    subset_tfidf = tfidf_matrix[df_subset.index]
    query_vec = tfidf_vec.transform([simple_preprocess(clean_query)])
    similarities = cosine_similarity(query_vec, subset_tfidf).flatten()
    df_subset['similarity_score'] = similarities

    # Sorting
    if similarities.max() == 0:
        results = df_subset.sort_values(by='rating', ascending=False)
    else:
        results = df_subset.sort_values(by=['similarity_score', 'rating'], ascending=[False, False])

    return results.head(top_n)

**SBERT Model**

In [36]:
from sentence_transformers import SentenceTransformer, util
import torch

model = SentenceTransformer('all-MiniLM-L6-v2')
#encoding and save
descriptions = df_cleaned['desc_1'].tolist()
dataset_embeddings = model.encode(descriptions, convert_to_tensor=True)
torch.save(dataset_embeddings, 'coffee_embeddings.pt')


In [37]:
model = SentenceTransformer('all-MiniLM-L6-v2')
dataset_embeddings = torch.load('coffee_embeddings.pt')
#Hard Filter
all_roasters = df_cleaned['roaster'].unique()
all_origins = set(df_cleaned['origin_1'].unique()) | set(df_cleaned['origin_2'].unique()) | set(df_cleaned['region'].unique())

In [63]:
def sbert_search(full_query, top_n=5):
    '''
    Hybrid Recommendation Engine (Advanced - SBERT version).

    Logic:
    1. Extract Hard Constraints (Price, Roaster, Location, Roast Level) from the natural language query.
    2. Clean the query to isolate flavor-related keywords.
    3. Filter the dataset to satisfy all Hard Constraints.
    4. Compute Semantic Similarity between query and coffee descriptions using Sentence-BERT embeddings.
    5. Return Top-N results based on the highest Semantic Similarity.
    '''
    query_lower = full_query.lower()

    max_price, p_text = extract_price_from_query(query_lower)
    roast_level, r_text = extract_roast_from_query(query_lower)

    clean_query = query_lower
    if p_text: clean_query = clean_query.replace(p_text, "")
    if r_text: clean_query = clean_query.replace(r_text, "")
    clean_query = clean_query.strip()

    filters = {'roaster': None, 'location': None}
    for r in all_roasters:
        if r.lower() in clean_query:
            filters['roaster'] = r
            break
    for geo in sorted(all_origins, key=len, reverse=True):
        if len(geo) > 3 and geo.lower() in clean_query:
            filters['location'] = geo
            break

    print(f" [SBERT Log] Entities: {filters} | Price: {max_price} | Roast: {roast_level}")


    mask = pd.Series([True] * len(df_cleaned))
    if filters['roaster']:
        mask &= df_cleaned['roaster'].str.contains(filters['roaster'], case=False, regex=False)
    if filters['location']:
        mask &= (df_cleaned['origin_1'].str.contains(filters['location'], case=False, regex=False) |
                 df_cleaned['region'].str.contains(filters['location'], case=False, regex=False))
    if max_price:
        mask &= (df_cleaned['100g_USD'] <= max_price)
    if roast_level:
        mask &= (df_cleaned['roast'] == roast_level)

    df_subset = df_cleaned[mask].copy()
    if df_subset.empty:
        return "No coffees match your specific combination of roast, price, and location."


    subset_indices = df_subset.index.tolist()
    subset_embeddings = dataset_embeddings[subset_indices]
    query_embedding = model.encode(clean_query, convert_to_tensor=True)
    cos_scores = util.cos_sim(query_embedding, subset_embeddings)[0]
    df_subset['similarity_score'] = cos_scores.cpu().numpy()

    results = df_subset.sort_values(by=['similarity_score', 'rating'], ascending=[False, False])

    return results.head(top_n)

**Evaluation**

In [112]:
import numpy as np
import warnings
from bert_score import score
warnings.filterwarnings("ignore")

def evaluate_top5(query):

    res_b = baseline_search(query, top_n=5)
    res_s = sbert_search(query, top_n=5)

    def analyze_model(res_df, model_name):



        names = res_df['name'].tolist()
        sim_scores = res_df['similarity_score'].tolist()
        descriptions = res_df['desc_1'].tolist()


        try:
            P, R, F1 = score(descriptions, [query] * len(descriptions), lang="en", verbose=False)
            bert_f1_list = F1.tolist()
        except Exception as e:
            bert_f1_list = [0.0] * len(sim_scores)

        print(f" {model_name} Top-5 :")
        print(f"{'Rank':<5} {'Coffee Name':<35} {'Sim Score ':<20} {'BERTScore ':<12}")
        print("-" * 95)

        for i in range(len(sim_scores)):
            coffee_name = str(names[i])[:33]
            print(f"{i+1:<5} {coffee_name:<35} {sim_scores[i]:.4f}               {bert_f1_list[i]:.4f}")


        avg_sim = np.mean(sim_scores)
        avg_bert = np.mean(bert_f1_list)


        print(f"   Average Similarity:  {avg_sim:.4f}")
        print(f"   Average BERTScore:   {avg_bert:.4f}")
        return avg_sim, avg_bert


    sim_b, bs_b = analyze_model(res_b, "Baseline (TF-IDF)")
    print("\n" + "." * 100 + "\n")
    sim_s, bs_s = analyze_model(res_s, "Advanced (SBERT)")



    improvement = ((sim_s - sim_b) / sim_b) * 100


    print(f" Improvement {improvement} %")




**1.Integrated Query (Hard Constraints + Semantic Search)**

In [114]:
evaluate_top5("Light roast coffee from Ethiopia under 20 dollars with bright citrus and floral notes")


 [Baseline Log] Entities: {'roaster': None, 'origin': 'Ethiopia'} | Price: 20.0 | Roast: Light
 [SBERT Log] Entities: {'roaster': None, 'location': 'Ethiopia'} | Price: 20.0 | Roast: Light
 Baseline (TF-IDF) Top-5 :
Rank  Coffee Name                         Sim Score            BERTScore   
-----------------------------------------------------------------------------------------------
1     Ethiopia Yirgacheffe G1             0.2901               0.8542
2     Ethiopia Bench Maji Natural Gesha   0.1386               0.8390
3     Ethiopia Limu Washed                0.1008               0.8296
4     Ethiopia Grade 1 Chelchele          0.0791               0.8306
5     Natural Sidamo Twakok G1            0.0728               0.8394
   Average Similarity:  0.1363
   Average BERTScore:   0.8385

....................................................................................................

 Advanced (SBERT) Top-5 :
Rank  Coffee Name                         Sim Score            BERTScor

In [115]:
print("baseline description")
baseline_search("Light roast coffee from Ethiopia under 20 dollars with bright citrus and floral notes")['desc_1'].tolist()

baseline description
 [Baseline Log] Entities: {'roaster': None, 'origin': None} | Price: 5.0 | Roast: Light


['High-toned, crisply sweet-tart. Concord grape, rhododendron, cocoa nib, sandalwood, maple syrup in aroma and cup. Brightly sweet structure with tart, juicy acidity; very plush, syrupy mouthfeel. The resonant finish consolidates to notes of cocoa nib and maple syrup with undertones of Concord grape.',
 'Resonant, sweet-savory, original. Baker’s chocolate, fragrant fresh-cut cedar, green grape, almond, wisteria in aroma and cup. Roundly tart, sweetly savory in structure; plushly syrupy mouthfeel. Flavor, particularly green grape and flowers, carries into a drying finish.',
 'Exceptionally balanced, high-toned. Pineapple, bergamot, cocoa nib, hazelnut, ginger flower in aroma and cup. Sweetly tart structure with cleanly fruity, vibrant acidity; very full, richly satiny mouthfeel. Crystalline finish that carries over all the aroma and flavor notes from the cup.',
 'Richly sweet-tart, floral-toned. Honeysuckle, mango, sandalwood, maple syrup, almond brittle in aroma and cup. Sweetly tart s

In [135]:
print("sbert description")
sbert_search("Light roast coffee from Ethiopia under 20 dollars with bright citrus and floral notes")['desc_1'].tolist()

sbert description
 [SBERT Log] Entities: {'roaster': None, 'location': 'Ethiopia'} | Price: 20.0 | Roast: Light


['Evaluated as espresso. Crisply sweet-tart, spice-toned. Baker’s chocolate, cardamom, tart cherry, fir, almond butter in aroma and small cup. Creamy body; bittersweet, wood-framed finish. Cardamom-toned chocolate blooms in three parts milk, along with sweet nut and tart fruit notes.',
 'Crisply sweet-tart, high-toned. Lemon verbena, cedar, pomegranate, lavender, maple syrup in aroma and cup. Sweetly tart in structure with bright, juicy acidity. The finish is deep and rich, with sweet, lemony herb tones, tart fruit notes, and hints of aromatic wood.',
 'Complex, multi-layered, deeply sweet. Lilac, cocoa nib, apricot, almond, tangerine zest in aroma and cup. Bright, juicy acidity; plush, syrupy mouthfeel. The resonant finish leads with notes of lilac and plum, balanced by suggestions of crisp cocoa.',
 'High-toned, richly sweet. Nectarine, apple blossom, redwood, dark chocolate, grapefruit zest in aroma and cup. Sweetly tart structure with bright, lively acidity; crisp, velvety-smooth m

**2.Short, potentially noisy queries**

In [117]:
evaluate_top5("floral jasmin")


 [Baseline Log] Entities: {'roaster': None, 'origin': None} | Price: None | Roast: None
 [SBERT Log] Entities: {'roaster': None, 'location': None} | Price: None | Roast: None
 Baseline (TF-IDF) Top-5 :
Rank  Coffee Name                         Sim Score            BERTScore   
-----------------------------------------------------------------------------------------------
1     Finca Lerida Gesha Natural          0.4090               0.7827
2     Costa Rica Central Valley Brandy    0.3949               0.7775
3     Honduras Relationship Coffee Norm   0.3830               0.7895
4     Panama Geisha                       0.3669               0.7748
5     Ethiopia Nano Genji #5              0.3629               0.7805
   Average Similarity:  0.3833
   Average BERTScore:   0.7810

....................................................................................................

 Advanced (SBERT) Top-5 :
Rank  Coffee Name                         Sim Score            BERTScore   
---------

In [120]:
print("baseline description")
baseline_search("floral jasmin")['desc_1'].tolist()

baseline description
 [Baseline Log] Entities: {'roaster': None, 'origin': None} | Price: None | Roast: None


['Richly sweet, floral-toned. Magnolia, date, almond, caramel, fresh-cut cedar in aroma and cup. Sweet structure with gentle, round acidity; full, syrupy-smooth mouthfeel. Floral and nut-toned finish.',
 'Richly sweet, floral-toned. Freesia-like flowers, orange zest, hazelnut, caramel, cedar in aroma and cup. Sweet structure with gentle, round acidity; full, syrupy-smooth mouthfeel. Crisp floral- and nut-toned finish.',
 'Richly floral-toned, deeply sweet. Jasmine, grapefruit zest, cocoa nib, marjoram, maple syrup in aroma and cup. Sweet-toned structure with juicy, bright acidity; syrupy-smooth mouthfeel. Balanced, floral, citrusy finish.',
 'Richly sweet, floral-toned. Honeysuckle, raisin, pistachio, caramel, fresh-cut cedar in aroma and cup. Sweet-toned structure with gentle, round acidity; velvety-smooth mouthfeel. Floral and nut-toned finish.',
 'Richly floral-toned, deeply sweet. Narcissus, Meyer lemon zest, cocoa nib, marjoram, maple syrup in aroma and cup. Sweet-toned structure 

In [118]:
print("sbert description")
sbert_search("floral jasmin")['desc_1'].tolist()

sbert description
 [SBERT Log] Entities: {'roaster': None, 'location': None} | Price: None | Roast: None


['High-toned, elegantly floral, intricately balanced. Concord grape, frankincense, aromatic orchid, candy cap mushroom, cocoa nib in aroma and cup. Deeply',
 'Floral, spice-toned. Jasmine, pink peppercorn, honey, apricot, myrrh in aroma and cup. Sweet structure with vivacious acidity; very full, syrupy mouthfeel. The finish consolidates to notes of pink peppercorn and honey.',
 'Floral and sweetly herbaceous with richly framed depth. Jasmine, spearmint, chocolate, watermelon, fine musk in aroma and cup. Deeply sweet structure with vibrant, juicy acidity; plush, syrupy mouthfeel. The resonant, flavor-saturated finish centers around chocolate and jasmine with musk undertones.',
 'Richly floral-toned, deeply sweet. Jasmine, grapefruit zest, cocoa nib, marjoram, maple syrup in aroma and cup. Sweet-toned structure with juicy, bright acidity; syrupy-smooth mouthfeel. Balanced, floral, citrusy finish.',
 'Rich-toned, deeply floral. Wisteria, cocoa nib, apricot, maple syrup, pink grapefruit ze

**3.Descriptive Query (Rich Sensory Profiles)**

In [132]:
evaluate_top5("very floral like a garden with some citrus fruit mixed in")

 [Baseline Log] Entities: {'roaster': None, 'origin': None} | Price: None | Roast: None
 [SBERT Log] Entities: {'roaster': None, 'location': None} | Price: None | Roast: None
 Baseline (TF-IDF) Top-5 :
Rank  Coffee Name                         Sim Score            BERTScore   
-----------------------------------------------------------------------------------------------
1     Ecuador La Hormiga “Naty”           0.2931               0.8227
2     Ethiopia Banko Gotiti Natural       0.2575               0.8174
3     Murang’a Kenya                      0.2078               0.8189
4     Estate-Grown Mamo                   0.1989               0.8186
5     Colombia Lagoon Java Anaerobic      0.1640               0.8173
   Average Similarity:  0.2243
   Average BERTScore:   0.8190

....................................................................................................

 Advanced (SBERT) Top-5 :
Rank  Coffee Name                         Sim Score            BERTScore   
---------

In [133]:
print("baseline description")
baseline_search("very floral like a garden with some citrus fruit mixed in")['desc_1'].tolist()

baseline description
 [Baseline Log] Entities: {'roaster': None, 'origin': None} | Price: None | Roast: None


['Delicately sweet-tart. Candy apple, green banana, grapefruit zest, brown sugar, marjoram in aroma and cup. Gently sweet structure with pert acidity; viscous, lively mouthfeel. The quiet finish consolidates to notes of tropical and citrus fruits and sweet garden herbs.',
 'High-toned, deeply sweet. Honeysuckle, blackberry, cardamom, tangerine zest, sandalwood in aroma and cup. Sweet-tart structure with vibrant acidity; buoyant, satiny mouthfeel. The flavor-saturated finish leads with rich floral notes and crisply sweet suggestions of mixed berries',
 'Richly and pungently zesty. Blackberry, baker’s chocolate, mixed peppercorn, violet, lemon zest in aroma and cup. Brisk and deeply rich-toned structure with a lemon and floral inflection; full, syrupy mouthfeel. Lemon and violet with a hint of cocoa in the long, dry finish. ',
 'Sweet-savory, richly wood-framed. Date, spice box, fresh-cut fir, cocoa powder, rhododendron in aroma and cup. Sweetly savory in structure with round, gentle aci

In [134]:
print("sbert description")
sbert_search("very floral like a garden with some citrus fruit mixed in")['desc_1'].tolist()

sbert description
 [SBERT Log] Entities: {'roaster': None, 'location': None} | Price: None | Roast: None


['Fruit-forward, delicate yet intense. Complex flowers (lily to lavendar), sandalwood, dark chocolate, plum, a hint of brandy in aroma and cup. Roundly and deeply crisp in structure; lightly satiny in mouthfeel. The finish is resonant and sweet with chocolate and a continuing hint of brandy.',
 'Citrusy and spice-toned. Lemon verbena, gardenia-like flowers, sandalwood, caramel, pink peppercorn in aroma and cup. Richly sweet-tart in structure with vibrant acidity; full, satiny mouthfeel. The finish consolidates to lemon verbena and sandalwood in the short, with caramel resurfacing in the long.',
 'Sweet, quietly intense, floral-toned. Rose, sandalwood, cashew, lemon verbena, a hint of mace in aroma and cup. Crisp and balanced in structure; silky mouthfeel. Flowers carry on deep into the finish.',
 'Richly sweet, floral-toned. Freesia-like flowers, orange zest, hazelnut, caramel, cedar in aroma and cup. Sweet structure with gentle, round acidity; full, syrupy-smooth mouthfeel. Crisp flor